In [ ]:

import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Introduction to RDDs")
    .config("spark.master", "local[*]")
    .getOrCreate()
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 16:54:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
sc = spark.sparkContext

In [ ]:

# 1 - parallelize an existing collection
numbers = [x for x in range(1000000)]
numbersRDD = sc.parallelize(numbers)

In [8]:


from dataclasses import dataclass

@dataclass
class StockValue:
    company: str
    date: str
    price: float

def read_stocks(filename: str):
    with open(filename, "r", encoding="utf-8") as f:
        next(f)
        return [
            StockValue(tokens[0], tokens[1], float(tokens[2]))
            for tokens in (line.strip().split(",") for line in f)
        ]

stocks_rdd = sc.parallelize(read_stocks("src/main/resources/data/stocks.csv"))

In [ ]:
# 2b - reading from files

stocksRDD2 = sc.textFile("src/main/resources/data/stocks.csv")

# notice to filter out header we cant drop 0 because its parallelized so we dont know which the header will be
stocksRDD2 = (
    stocksRDD2.map(lambda line : line.split(","))
    .filter(lambda tokens: tokens[0].toUpperCase() == tokens[0]) # filter out the header
    .map(lambda tokens : StockValue(tokens[0], tokens[1], float(tokens[2])))
)

In [13]:
# read from a dataframe


stocksDF = spark.read.option("header", "true").csv("src/main/resources/data/stocks.csv")

# RDD of for
stocksRDD3 = stocksDF.rdd

# or - RDD of type StockValue
stocksRDD4 = stocksDF.rdd.map(lambda row:  StockValue(row["company"], row["date"], row["price"])) 

In [16]:
# rdd to a dataframe
numbersDF = numbersRDD.map(lambda x: Row(numbers=x)).toDF()